# OnionEye · 03 · Build Dataset 2 (defect crops)
Uses the trained finder to cut every onion out of the defect photos and label it from the damage boxes inside it. GPU helps (T4), ~20 min.

Output: `MyDrive/onioneye/defect_dataset.zip` + one review sheet per class in `reports/`. **Look at the sheets** and list any wrong crops in step 4.

In [ ]:
!pip -q install ultralytics opencv-python-headless

In [ ]:
# ---- setup: works on Google Colab (default) and Kaggle
import os, sys, glob, shutil, zipfile, random
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WORK = '/content/drive/MyDrive/onioneye', '/content'
elif IN_KAGGLE:
    ROOT, WORK = '/kaggle/working/onioneye', '/kaggle/temp'
else:
    ROOT, WORK = os.path.abspath('onioneye_data'), os.path.abspath('work')
for d in (ROOT, WORK, f'{ROOT}/models', f'{ROOT}/reports'):
    os.makedirs(d, exist_ok=True)
print('saving to', ROOT)

In [ ]:
# the shared data code lives in your Drive: MyDrive/onioneye/onioneye_data.py (copy it from the repo's ml/ folder)
shutil.copy(f'{ROOT}/onioneye_data.py', 'onioneye_data.py')
import importlib, onioneye_data as od
importlib.reload(od)

In [ ]:
import onioneye_data as od
RAW = f'{WORK}/raw_defects'; CLS = f'{WORK}/defects'; HOLD = f'{WORK}/mendeley_unhealthy_holdout'
if not os.path.exists(RAW): zipfile.ZipFile(f'{ROOT}/raw_defect_sources.zip').extractall(RAW)
shutil.rmtree(CLS, ignore_errors=True)
from ultralytics import YOLO
finder = YOLO(f'{ROOT}/models/finder_best.pt')
ONION = [k for k, v in finder.names.items() if v == 'onion'][0]
def find_onions(img, conf=0.4):
    r = finder.predict(img, imgsz=640, conf=conf, verbose=False)[0]
    return [tuple(b) for b, c in zip(r.boxes.xyxy.cpu().numpy().tolist(), r.boxes.cls.cpu().numpy()) if int(c) == ONION]

## 1 · Crops from the box datasets (vcsiy, ciqkj, urad) and the folder dataset

In [ ]:
import pandas as pd
from collections import Counter
total = Counter()
for src in od.DEFECT_BOX_SOURCES:
    if not os.path.exists(f'{RAW}/{src["name"]}'): print('missing', src['name']); continue
    st, skipped = od.build_defect_from_boxes(src, RAW, CLS, find_onions)
    print(src['name'], dict(sum((Counter({k[1]: v}) for k, v in st.items()), Counter())), 'skipped:', dict(skipped))
    total.update(st)
for src in od.DEFECT_FOLDER_SOURCES:
    if os.path.exists(f'{RAW}/{src["name"]}'):
        st = od.build_defect_from_folders(src, RAW, CLS); total.update(st); print(src['name'], dict(st))

## 2 · Mendeley healthy bulbs → `good` (capped), unhealthy → held out for a check in notebook 04

In [ ]:
MX = f'{WORK}/mendeley'
if not os.path.exists(MX):
    zipfile.ZipFile(f'{ROOT}/mendeley_onion.zip').extractall(MX)
    for inner in glob.glob(f'{MX}/**/*.zip', recursive=True):
        zipfile.ZipFile(inner).extractall(os.path.dirname(inner))
st, n_bad = od.build_defect_from_mendeley(MX, CLS, holdout_dir=HOLD); total.update(st)
print('mendeley good:', sum(st.values()), '| unhealthy held out:', len(os.listdir(HOLD)), 'of', n_bad)
df = pd.Series(total).unstack(0).reindex(od.DEFECT_CLASSES).fillna(0).astype(int)
df['total'] = df.sum(axis=1); display(df)

## 3 · Review sheets: 48 random crops per class (saved to `reports/`)

In [ ]:
from IPython.display import Image
for c in od.DEFECT_CLASSES:
    files = sorted(glob.glob(f'{CLS}/*/{c}/*.jpg'))
    pick = random.Random(0).sample(files, min(48, len(files)))
    p = od.contact_sheet(pick, f'{ROOT}/reports/defect_{c}.png', cols=8, cell=150, title=f'{c}: {len(files)} crops')
    print(c); display(Image(p))
    open(f'{ROOT}/reports/defect_{c}_files.txt', 'w').write('\n'.join(os.path.basename(x) for x in pick))

## 4 · Remove wrong crops (optional)
Find a bad crop's position on a sheet (row by row, starting at 0), look up its name in `reports/defect_<class>_files.txt`, paste names below, run.

In [ ]:
BAD = [
    # 'vcsiy__IMG-20250310-WA0381_jpg__0.jpg',
]
for name in BAD:
    for p in glob.glob(f'{CLS}/*/*/{name}'): os.remove(p); print('removed', p)

## 5 · Balance and save

In [ ]:
# keep 'good' at most 3x the biggest defect class, so the model doesn't just say 'good'
counts = {c: len(glob.glob(f'{CLS}/train/{c}/*')) for c in od.DEFECT_CLASSES}
cap = 3 * max(v for k, v in counts.items() if k != 'good')
goods = sorted(glob.glob(f'{CLS}/train/good/*')); random.Random(1).shuffle(goods)
for p in goods[cap:]: os.remove(p)
for sp in ['train', 'val', 'test']:                     # every class folder must exist in every split
    for c in od.DEFECT_CLASSES: os.makedirs(f'{CLS}/{sp}/{c}', exist_ok=True)
print({sp: {c: len(os.listdir(f'{CLS}/{sp}/{c}')) for c in od.DEFECT_CLASSES} for sp in ['train', 'val', 'test']})
shutil.make_archive(f'{ROOT}/defect_dataset', 'zip', CLS)
shutil.make_archive(f'{ROOT}/mendeley_unhealthy_holdout', 'zip', HOLD)

Done ✅ Next: **04_train_classifier**.